# 🏏 Cricket Rules RAG Chatbot — Colab runner

**Runtime → Change runtime type → T4 GPU** first (the generator is much faster on GPU).
Run the cells in order. Steps 3–6 build the corpus and index; step 8 runs the evaluation once your team has written `eval/golden_set.jsonl`; step 10 launches the chatbot with a public `gradio.live` link.

## 1. Get the code

In [ ]:
!rm -rf /content/MSAI-633-M50
!git clone -b claude/amazing-fermat-bu68c9 https://github.com/vnaiduyerra/MSAI-633-M50 /content/MSAI-633-M50
%cd /content/MSAI-633-M50

## 2. Install and run the offline tests
If an earlier run of this notebook installed different package versions, first use **Runtime → Disconnect and delete runtime**, then start again from step 1.

In [ ]:
!pip install -q -r requirements.txt
!python -m unittest discover -s tests
!python -c "import numpy, transformers, sentence_transformers, gradio, faiss; print('numpy', numpy.__version__, '| transformers', transformers.__version__, '| sentence-transformers', sentence_transformers.__version__, '| gradio', gradio.__version__)"

## 3. Crawl (≈1 request/second)
Put a real contact e-mail here; it goes into the User-Agent as the spec requires.

In [ ]:
import os
os.environ['CRAWL_CONTACT_EMAIL'] = 'your.name@ucumberlands.edu'  # <-- change me
!python -m rag.cli crawl

## 4. Extract, 5. Chunk

In [ ]:
!python -m rag.cli extract
!python -m rag.cli chunk

## 6. Index fixed chunks, then migrate to structure-aware chunks (re-chunk + orphan check)

In [ ]:
!python -m rag.cli index --strategy fixed
!python -m rag.cli index --strategy structure
!python -m rag.cli verify-metric

## 7. Golden set
Write `eval/golden_set.jsonl` (start from `eval/golden_set.template.jsonl`). Use the search helper to find the exact page and an evidence quote:

In [ ]:
!python -m rag.cli search "penalty runs ball tampering"

In [ ]:
# after uploading eval/golden_set.jsonl:
!python -m rag.cli golden --assign-split

## 8. Evaluate on dev, probe injection, export failures

In [ ]:
!python -m rag.cli eval --split dev
!python -m rag.cli injection-probe
!python -m rag.cli failures --config B3 --split dev

## 9. Final: test split ONCE, then generate the report tables

In [ ]:
# !python -m rag.cli eval --split test   # uncomment only when the configuration is frozen
!python -m rag.cli report

Download the outputs to commit them from your own account:

In [ ]:
!pip freeze > requirements.lock.txt   # exact versions behind the reported numbers
!zip -qr /content/outputs.zip data/processed eval reports requirements.lock.txt
from google.colab import files; files.download('/content/outputs.zip')

## 10. Launch the chatbot
Open the `https://….gradio.live` link printed below. Stop this cell to shut it down.

In [ ]:
!python app.py